# AAM — KPIs selecionados para dados sintéticos

Modelo focado nos dois KPIs de AAM priorizados para o artigo 2: **AAM-CAP-01** (Taxa de Acomodação Integrada da Demanda) e **AAM-PRED-01** (Pontualidade Operacional). Avaliados no escopo do ConOps UAM v2.0 da FAA, com a UAM como subconjunto urbano da AAM.

Execute as células **em ordem**: primeiro o núcleo comum, depois o modelo focado. Fórmulas, τ e Δ são **propostas** para o artigo 2.

## Núcleo comum

Idêntico ao `KPI_Common.ipynb`.

In [ ]:
package KPI_Taxonomy {
    doc /* Enumerations and metadata used to classify KPAs, performance objectives and KPI candidates (article 1: applicability, analytical status, measurement maturity, taxonomic fit). */
    private import ScalarValues::*;

    enum def Domain { enum UTM; enum AAM; }
    enum def Applicability { enum Applicable; enum ConditionallyApplicable; enum NotApplicable; }
    enum def RetentionStatus { enum ProvisionallyRetained; enum ConditionallyRetained; }
    enum def Operationalization {
        doc /* Measurement maturity of the candidate, kept distinct from retention status (article 1). */
        enum ConceptualFormulaInArticle1;
        enum ProposedInThisStudy;
        enum Pending;
    }
    enum def TaxonomicFit { enum Direct; enum Provisional; enum Gap; }

    metadata def KPAApplicability {
        attribute domain : Domain;
        attribute applicability : Applicability;
    }
    metadata def CandidateKPI {
        attribute code : String;
        attribute domain : Domain;
        attribute measurementConstruct : String;
        attribute status : RetentionStatus;
        attribute maturity : Operationalization;
    }
    metadata def DomainExtension {
        doc /* Construct that extends the reference KPA taxonomy for the domain. */
        attribute fit : TaxonomicFit;
    }
    metadata def SupportingDiagnosticMetric {
        doc /* Explains why a system-level KPI behaves as observed; not itself a system-level KPI. */
    }

    enum def SourceClass {
        doc /* Where a KPI input exists according to the FAA concepts of operations (experiment E1).
         * NetworkShared: exchanged across the USS Network or the federated service network during operations.
         * ProviderArchived: retained by USSs, PSUs or operators and available on authorized request.
         * InternallyHeld: exists within one operator or provider, with no sharing or archiving requirement.
         * NotAddressed: no corresponding data element in the concept of operations. */
        enum NetworkShared;
        enum ProviderArchived;
        enum InternallyHeld;
        enum NotAddressed;
    }
    enum def InputRole {
        doc /* Principal: the quantity the KPI counts or measures; Supporting: any other input. */
        enum Principal;
        enum Supporting;
    }
    attribute def KPIInput {
        doc /* A quantity consumed by a KPI candidate, described in its doc and classified by @DataSource. */
    }
    metadata def DataSource {
        doc /* Source class of a data element in the FAA concepts of operations, with the section and printed page. */
        attribute sourceClass : SourceClass;
        attribute conopsRef : String;
    }
    metadata def KPIInputRole {
        doc /* Role of an input in a KPI; on a KPIInput attribute, or on a dependency from a KPI to a data-item attribute. */
        attribute role : InputRole;
    }
}

In [ ]:
package ICAO_KPAs {
    doc /* The eleven ICAO ATM reference KPAs (ICAO Doc 9854 and Doc 9883), used as the common performance reference structure. Performance Intents: article 1, Table 1. */

    concern def AccessAndEquity { doc /* ICAO reference KPA: Access and Equity. */ }
    concern def Capacity { doc /* ICAO reference KPA: Capacity. */ }
    concern def CostEffectiveness { doc /* ICAO reference KPA: Cost Effectiveness. */ }
    concern def Efficiency { doc /* ICAO reference KPA: Efficiency. */ }
    concern def Environment { doc /* ICAO reference KPA: Environment. */ }
    concern def Flexibility { doc /* ICAO reference KPA: Flexibility. */ }
    concern def GlobalInteroperability { doc /* ICAO reference KPA: Global Interoperability. */ }
    concern def Participation { doc /* ICAO reference KPA: ATM Community Participation. */ }
    concern def Predictability { doc /* ICAO reference KPA: Predictability. */ }
    concern def Safety { doc /* ICAO reference KPA: Safety. */ }
    concern def Security { doc /* ICAO reference KPA: Security. */ }
}

In [ ]:
package System_Boundaries {
    doc /* Selected system boundaries (article 1, Sec. 2.4). */

    part def UTMSystem {
        doc /* Based on the FAA UTM ConOps v2.0: traffic-management services, operational coordination,
         * information exchange, service interactions and constraint management that support
         * unmanned operations. Intrinsic aircraft or propulsion characteristics are excluded. */
    }
    part def AAMSystem {
        doc /* Based on the FAA UAM ConOps v2.0 and AAM Implementation Plan: an integrated mobility
         * system of systems with aircraft operations, traffic-management interactions, vertiports,
         * supporting infrastructure, energy, digital services, ground access, users and affected
         * communities. UAM is treated as the urban subset of AAM. */
    }
}

In [ ]:
package KPI_Patterns {
    doc /* Common calculation patterns shared by UTM and AAM candidates. PROPOSED for article 2; not defined in article 1. */
    private import ScalarValues::*;

    abstract calc def AccommodationRate {
        doc /* Accommodated demand over valid demand in a period and volume, in percent. */
        in accommodated : Natural;
        in validDemand : Natural;
        return rate : Real = accommodated / validDemand * 100;
    }
    abstract calc def TemporalConformanceRate {
        doc /* Completed operations whose actual time deviates from the planned time by no more
         * than the tolerance, over all completed operations, in percent. */
        in withinTolerance : Natural;
        in completed : Natural;
        in toleranceSeconds : Real;
        return rate : Real = withinTolerance / completed * 100;
    }
}

## AAM_Selected

O modelo focado traz a arquitetura mínima do ConOps, os itens de dados que alimentam os KPIs, os parâmetros em aberto, os dois objetivos de desempenho, os dois KPIs, as métricas diagnósticas, a rastreabilidade até os atributos de dados e o `analysis def` de uma rodada sintética.

In [ ]:
package AAM_Selected {
    doc /* Focused model for the two AAM KPIs prioritized for synthetic data in article 2:
     * AAM-CAP-01 (Integrated Demand Accommodation Rate) and AAM-PRED-01 (Operational Punctuality).
     * Evaluated within the scope of the FAA UAM ConOps v2.0 (UAM as the urban subset of AAM).
     * Only the elements needed to compute the two KPIs are modeled (minimum-sufficiency criterion).
     * Formulas and parameters are PROPOSED for article 2. */
    private import ScalarValues::*;
    private import ICAO_KPAs::*;
    private import KPI_Taxonomy::*;
    private import KPI_Patterns::*;
    private import System_Boundaries::*;

    // ---------- Architecture (FAA UAM ConOps v2.0, Sec. 4.3 and Sec. 5) ----------

    part def UAMOperator {
        doc /* UAM operator: conducts scheduled or on-demand operations from customer requests and submits
         * the desired UAM Operational Intent to its PSU (UAM ConOps Sec. 4.3.2, p. 13). */
        ref item flightRecords : FlightRecord[0..*];
    }
    part def PSU {
        doc /* Provider of Services for UAM: confirms that a submitted UAM Operational Intent is complete and
         * strategically deconflicted, shares it with the federated service network, and archives
         * operational data (UAM ConOps Sec. 4.3.4, p. 14). */
        ref item receivedRequests : TripRequest[0..*];
    }
    part def FederatedServiceNetwork {
        doc /* Federated service network: the collection of connected PSUs sharing subscriber, FAA and
         * supplemental data (UAM ConOps Sec. 4.3.5, p. 15). */
    }
    part def FAAIndustryDataExchange {
        doc /* FAA-Industry Data Exchange Protocol: interface through which the FAA requests UAM operational
         * data on demand and sends FAA information to the federated service network (UAM ConOps
         * Sec. 4.3.1.3, p. 12; Sec. 5, p. 23). The UAM ConOps does not use the term FIMS. */
    }
    part def Vertiport {
        doc /* UAM vertiport: provides current and future resource availability (e.g., open/closed, pad
         * availability) to the network (UAM ConOps Sec. 4.3.7, p. 15). Chargers are an assumption of the
         * synthetic study; charging is not addressed in the ConOps. */
        attribute padCount : Natural { @DataSource { sourceClass = SourceClass::NetworkShared; conopsRef = "UAM §4.3.7, p. 15"; } }
        attribute chargerCount : Natural { @DataSource { sourceClass = SourceClass::NotAddressed; conopsRef = "- (not addressed)"; } }
    }
    part def UAMAircraft {
        attribute available : Boolean { @DataSource { sourceClass = SourceClass::NotAddressed; conopsRef = "- (not addressed)"; } }
    }
    part def UAMCorridor {
        doc /* UAM Corridor; availability (open/closed) is set by ATC (UAM ConOps Sec. 4.3.1.2, p. 12). */
        attribute open : Boolean { @DataSource { sourceClass = SourceClass::NetworkShared; conopsRef = "UAM §4.3.1, p. 12; §4.4, p. 17"; } }
    }
    part def AAMNetwork :> AAMSystem {
        doc /* Instance of the selected AAM boundary used for the synthetic study. */
        part operators : UAMOperator[1..*];
        part psuProviders : PSU[1..*];
        part network : FederatedServiceNetwork[1];
        part dataExchange : FAAIndustryDataExchange[1];
        part vertiports : Vertiport[2..*];
        part fleet : UAMAircraft[1..*];
        part corridors : UAMCorridor[1..*];
    }

    // ---------- Data items consumed by the KPIs ----------

    enum def ServiceOutcome {
        doc /* Confirmed: the PSU confirmed the UAM Operational Intent (UAM ConOps Sec. 4.3.4, p. 14).
         * DeniedNoPad and DeniedAirspace map to vertiport resources and airspace or corridor constraints
         * named in the ConOps; DeniedNoCharge and DeniedNoAircraft are operator-internal causes that the
         * ConOps does not address. */
        enum Confirmed;
        enum DeniedNoPad;
        enum DeniedNoCharge;
        enum DeniedNoAircraft;
        enum DeniedAirspace;
        enum DeniedInvalid;
    }
    item def TripRequest {
        doc /* Customer request for a trip between two vertiports, received by the UAM operator (UAM ConOps
         * Sec. 4.3.2, p. 13; Sec. 6.1.1, p. 25). scheduledDeparture is the departure time of the
         * confirmed UAM Operational Intent (Sec. 3.2, p. 8); it is set only for confirmed requests. Internally held by the operator: requests that are not
         * served never become UAM Operational Intents. Times in seconds from the start of the period. */
        attribute requestId : String;
        attribute isValid : Boolean { @DataSource { sourceClass = SourceClass::InternallyHeld; conopsRef = "UAM §4.3.2, p. 13; §6.1.1, p. 25"; } }
        attribute outcome : ServiceOutcome { @DataSource { sourceClass = SourceClass::NetworkShared; conopsRef = "UAM §4.3.4, p. 14 (items 3-4)"; } }
        attribute requestedDeparture : Real { @DataSource { sourceClass = SourceClass::InternallyHeld; conopsRef = "UAM §4.3.2, p. 13; §6.1.1, p. 25"; } }
        attribute scheduledDeparture : Real { @DataSource { sourceClass = SourceClass::NetworkShared; conopsRef = "UAM §3.2, p. 8"; } }
    }
    item def FlightRecord {
        doc /* Record of a completed flight. Scheduled times are the key operational event times of the
         * confirmed UAM Operational Intent (UAM ConOps Sec. 3.2, p. 8); actual times come from operational
         * data archived by the PSU (Sec. 4.3.4, p. 14, item 8; Sec. 6.1.3, p. 26). */
        attribute requestId : String;
        attribute scheduledDeparture : Real;
        attribute scheduledArrival : Real { @DataSource { sourceClass = SourceClass::NetworkShared; conopsRef = "UAM §3.2, p. 8"; } }
        attribute actualDeparture : Real;
        attribute actualArrival : Real { @DataSource { sourceClass = SourceClass::ProviderArchived; conopsRef = "UAM §6.1.2, p. 25; §4.3.4, p. 14 (item 8)"; } }
    }

    // ---------- Measurement parameters (open decisions, section 3.4 of the proposal) ----------

    attribute def MeasurementParameters {
        doc /* periodSeconds = aggregation period T; toleranceSeconds = tau for AAM-PRED-01, applied to
         * arrival time; shiftWindowSeconds = Delta, the departure shift still counted as served. */
        attribute periodSeconds : Real;
        attribute toleranceSeconds : Real;
        attribute shiftWindowSeconds : Real;
    }

    // ---------- Performance Objectives (article 1, Table 4) ----------

    requirement def PO_Capacity {
        doc /* Accommodate transportation demand within the integrated AAM operational system. */
        @KPAApplicability { domain = Domain::AAM; applicability = Applicability::Applicable; }
        subject net : AAMNetwork;
        frame concern kpa : Capacity;
    }
    requirement def PO_Predictability {
        doc /* Provide predictable operational outcomes with respect to planned timing. */
        @KPAApplicability { domain = Domain::AAM; applicability = Applicability::Applicable; }
        subject net : AAMNetwork;
        frame concern kpa : Predictability;
    }

    // ---------- KPI candidates ----------

    calc def <'AAM-CAP-01'> IntegratedDemandAccommodationRate :> AccommodationRate {
        doc /* accommodated = valid requests with outcome Confirmed (pad, charge, aircraft and corridor all
         * available) and scheduledDeparture - requestedDeparture no greater than Delta; validDemand =
         * valid requests in the period. PROPOSED. */
        @CandidateKPI {
            code = "AAM-CAP-01";
            domain = Domain::AAM;
            measurementConstruct = "Integrated Demand Accommodation";
            status = RetentionStatus::ProvisionallyRetained;
            maturity = Operationalization::ProposedInThisStudy;
        }
    }
    calc def <'AAM-PRED-01'> OperationalPunctuality :> TemporalConformanceRate {
        doc /* withinTolerance = completed flights with |actualArrival - scheduledArrival| no greater
         * than tau; completed = flights with a record in the period. PROPOSED. */
        @CandidateKPI {
            code = "AAM-PRED-01";
            domain = Domain::AAM;
            measurementConstruct = "Temporal Predictability";
            status = RetentionStatus::ProvisionallyRetained;
            maturity = Operationalization::ProposedInThisStudy;
        }
    }

    // ---------- Supporting diagnostic metrics (AAM-CAP-01: article 1, Sec. 4.5) ----------

    calc def VertiportThroughput { doc /* Diagnostic of AAM-CAP-01. */ @SupportingDiagnosticMetric; return metricValue : Real; }
    calc def PadUtilization { doc /* Diagnostic of AAM-CAP-01. */ @SupportingDiagnosticMetric; return metricValue : Real; }
    calc def ChargingAvailability { doc /* Diagnostic of AAM-CAP-01. */ @SupportingDiagnosticMetric; return metricValue : Real; }
    calc def AirspaceConstraints { doc /* Diagnostic of AAM-CAP-01. */ @SupportingDiagnosticMetric; return metricValue : Real; }
    calc def FleetAvailability { doc /* Diagnostic of AAM-CAP-01. */ @SupportingDiagnosticMetric; return metricValue : Real; }
    calc def DelayByCause {
        doc /* Share of late arrivals per cause (charging wait, pad occupancy, corridor restriction).
         * Diagnostic of AAM-PRED-01; proposed for article 2. */
        @SupportingDiagnosticMetric;
        return metricValue : Real;
    }

    // ---------- Traceability ----------

    dependency cap_measures_po from IntegratedDemandAccommodationRate to PO_Capacity;
    dependency pred_measures_po from OperationalPunctuality to PO_Predictability;

    dependency cap_uses_validity from IntegratedDemandAccommodationRate to TripRequest::isValid { @KPIInputRole { role = InputRole::Supporting; } }
    dependency cap_uses_outcome from IntegratedDemandAccommodationRate to TripRequest::outcome { @KPIInputRole { role = InputRole::Principal; } }
    dependency cap_uses_requested from IntegratedDemandAccommodationRate to TripRequest::requestedDeparture { @KPIInputRole { role = InputRole::Supporting; } }
    dependency cap_uses_scheduled from IntegratedDemandAccommodationRate to TripRequest::scheduledDeparture { @KPIInputRole { role = InputRole::Supporting; } }
    dependency cap_uses_pads from IntegratedDemandAccommodationRate to Vertiport::padCount { @KPIInputRole { role = InputRole::Supporting; } }
    dependency cap_uses_chargers from IntegratedDemandAccommodationRate to Vertiport::chargerCount { @KPIInputRole { role = InputRole::Supporting; } }
    dependency cap_uses_fleet from IntegratedDemandAccommodationRate to UAMAircraft::available { @KPIInputRole { role = InputRole::Supporting; } }
    dependency cap_uses_corridor from IntegratedDemandAccommodationRate to UAMCorridor::open { @KPIInputRole { role = InputRole::Supporting; } }
    dependency pred_uses_scheduled from OperationalPunctuality to FlightRecord::scheduledArrival { @KPIInputRole { role = InputRole::Principal; } }
    dependency pred_uses_actual from OperationalPunctuality to FlightRecord::actualArrival { @KPIInputRole { role = InputRole::Supporting; } }

    dependency throughput_explains from VertiportThroughput to IntegratedDemandAccommodationRate;
    dependency pads_explains from PadUtilization to IntegratedDemandAccommodationRate;
    dependency charging_explains from ChargingAvailability to IntegratedDemandAccommodationRate;
    dependency airspace_explains from AirspaceConstraints to IntegratedDemandAccommodationRate;
    dependency fleet_explains from FleetAvailability to IntegratedDemandAccommodationRate;
    dependency delay_explains from DelayByCause to OperationalPunctuality;

    // ---------- KPI targets as verifiable requirements ----------

    requirement def CapacityTarget :> PO_Capacity {
        doc /* Refines PO_Capacity into a verifiable statement on AAM-CAP-01: the measured value shall be at
         * least the target. Target values are illustrative assumptions of the synthetic study. */
        attribute measured : Real;
        attribute target : Real;
        require constraint { measured >= target }
    }
    requirement def PredictabilityTarget :> PO_Predictability {
        doc /* Refines PO_Predictability into a verifiable statement on AAM-PRED-01. */
        attribute measured : Real;
        attribute target : Real;
        require constraint { measured >= target }
    }
    dependency capTarget_on_kpi from CapacityTarget to IntegratedDemandAccommodationRate;
    dependency predTarget_on_kpi from PredictabilityTarget to OperationalPunctuality;

    // ---------- Synthetic study ----------

    part def SyntheticScenario {
        doc /* Configuration of one synthetic run (experiments E2 to E4 of the proposal). */
        attribute vertiportCount : Natural;
        attribute padsPerVertiport : Natural;
        attribute chargersPerVertiport : Natural;
        attribute fleetSize : Natural;
        attribute peakDemandPerHour : Real;
        attribute turnaroundSeconds : Real;
        attribute chargingSeconds : Real;
        attribute corridorClosureProbability : Real;
        attribute params : MeasurementParameters;
    }
    analysis def CapacityPredictabilityRun {
        doc /* One synthetic run: the generator produces requests and flight records for the scenario;
         * the counts below are taken from them and both KPIs are evaluated. */
        subject scenario : SyntheticScenario;
        objective {
            doc /* Observe AAM-CAP-01 and AAM-PRED-01 under the scenario configuration. */
        }
        in nConfirmed : Natural;
        in nValid : Natural;
        in nWithinTolerance : Natural;
        in nCompleted : Natural;
        out capResult : Real = IntegratedDemandAccommodationRate(nConfirmed, nValid);
        out predResult : Real = OperationalPunctuality(nWithinTolerance, nCompleted, scenario.params.toleranceSeconds);
    }

    // ---------- Base case: analysis result bound to the targets ----------

    part baseCase : AAMNetwork;
    analysis baseRun : CapacityPredictabilityRun;
    requirement baseCaseCapacity : CapacityTarget {
        attribute :>> measured = baseRun.capResult;
        attribute :>> target = 80.0;
    }
    requirement baseCasePredictability : PredictabilityTarget {
        attribute :>> measured = baseRun.predResult;
        attribute :>> target = 90.0;
    }
    satisfy baseCaseCapacity by baseCase;
    satisfy baseCasePredictability by baseCase;
}

## Visualizações

In [ ]:
%viz AAM_Selected::AAMNetwork

In [ ]:
%viz AAM_Selected::PO_Capacity

In [ ]:
%viz AAM_Selected::IntegratedDemandAccommodationRate

In [ ]:
%viz AAM_Selected::CapacityPredictabilityRun

In [ ]:
%viz AAM_Selected::CapacityTarget

In [ ]:
%viz AAM_Selected::baseCaseCapacity